# JAX AUTOGRAD ADVANTAGES

In [39]:
import jax
import jax.numpy as jnp
from flax import linen as nn
import optax

from sklearn.model_selection import train_test_split
import pandas as pd

# prep
data = pd.read_csv('polyx0data.csv')
X = data['Coefficients']
y = data['Starting Point']
X = X.apply(eval)

In [40]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train, X_test = jnp.array(X_train.tolist()), jnp.array(X_test.tolist())
y_train, y_test = jnp.expand_dims(jnp.array(y_train.tolist()), axis=1), jnp.expand_dims(jnp.array(y_test.tolist()), axis=1)

In [41]:
class StartingPointNN(nn.Module):
	def setup(self):
		self.dense1 = nn.Dense(16)
		self.dense2 = nn.Dense(4)
		self.dense3 = nn.Dense(1)

	def __call__(self, x):
		x = self.dense1(x)
		x = nn.relu(x)
		x = self.dense2(x)
		x = nn.relu(x)
		x = self.dense3(x)
		return x


@jax.jit
def newton_solver(coeffs, x0, max_iter=10):
	def f(x):
		return jnp.squeeze(jnp.polyval(coeffs, x))
	x = x0
	for _ in range(max_iter):
		fx = f(x)
		dfx = jax.grad(f)(x)
		x = x - fx / dfx
	return x

# loss function (using custom newton solver)
@jax.jit
def compute_loss(params, X, y):
	x0s = model.apply({'params': params}, X)
	roots_pred = jax.vmap(newton_solver)(X, x0s)
	roots_label = jax.vmap(newton_solver)(X, y)
	loss = jnp.mean((roots_pred - roots_label) ** 2)
	return loss

# update params
#@jax.jit
def update(params, opt_state, X, y):
	compute_loss_grad = jax.grad(compute_loss)
	grads = compute_loss_grad(params, X, y)
	updates, new_opt_state = optimizer.update(grads, opt_state)
	new_params = optax.apply_updates(params, updates)
	return new_params, new_opt_state

In [42]:
# define
model = StartingPointNN()
params = model.init(jax.random.PRNGKey(0), X_train[:1])['params']
optimizer = optax.adam(learning_rate=0.01)
opt_state = optimizer.init(params)

# train
epochs = 90
for epoch in range(epochs):
    params, opt_state = update(params, opt_state, X_train, y_train)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        loss = compute_loss(params, X_train, y_train)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss:.4f}")

Epoch [1/90], Loss: 23.7874
Epoch [10/90], Loss: 50.3509
Epoch [20/90], Loss: 773.2328
Epoch [30/90], Loss: 30.9886
Epoch [40/90], Loss: 163.5868
Epoch [50/90], Loss: 18.1520
Epoch [60/90], Loss: 1927.0009
Epoch [70/90], Loss: 12.3767
Epoch [80/90], Loss: 28.9443
Epoch [90/90], Loss: 15.0518


In [43]:
# evaluate
y_pred = model.apply({'params': params}, X_test)
roots_pred = jax.vmap(newton_solver)(X_test, y_pred)
roots_test = jax.vmap(newton_solver)(X_test, y_test)
accuracy = jnp.mean(jnp.isclose(roots_pred, roots_test, atol=1e-2))
for r1, r2 in zip(roots_pred, roots_test):
	print(f'{r1.item(), r2.item()}')
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")


(-0.6382162570953369, -0.6382162570953369)
(-1.1401771306991577, 0.9990730881690979)
(-0.850961446762085, -0.8509613871574402)
(1.0212676525115967, 1.0212676525115967)
(-1.2269690036773682, 0.17665636539459229)
(7.281091213226318, 6.0015363693237305)
(-0.5883980989456177, -0.5883980989456177)
(0.7812405228614807, 0.7813555002212524)
(0.7489582300186157, 0.7489582300186157)
(-0.41510993242263794, 1.6878092288970947)
(-0.22161583602428436, -0.22161586582660675)
(-0.5001869201660156, -0.5001869201660156)
(0.589625895023346, 0.589625895023346)
(-0.4355008602142334, 1.6477984189987183)
(3.252087354660034, 3.332263231277466)
(-0.07460617274045944, -0.07460617274045944)
(-0.7655990123748779, -0.7655990719795227)
(-0.9941397309303284, 1.3276894092559814)
(-4.9524712562561035, 4.215551376342773)
(-0.6368350386619568, 2.4357540607452393)

Test Accuracy: 55.00%
